# PCI on Pearl's Desert Traveler — Verification

> **Goal.** Reproduce numerically the claims in
> `claude_notes/2026-05-26_pearl_paac_proposal.md`:
>
> 1. On Pearl's original desert traveler, PCI's necessity expectation under
>    uniform $\Gamma$ and uniform $u$ prior is $\tfrac{1}{8}$ for each cause
>    (singleton $\mathbf{C}=\{X\}$ or $\mathbf{C}=\{P\}$), with $\mathbf{T}$
>    ranging over subsets of $\{c, d\}$.
> 2. After forensic conditioning on *no cyanide* ($u=1$), the necessity
>    expectation becomes $\tfrac{1}{4}$ for the shooter, $0$ for the poisoner.
> 3. In the weak-poison variant, Pearl's $\mathrm{PS}$ values are
>    $\mathrm{PS}(P) = 0.1$ and $\mathrm{PS}(X) \approx 0.76$.
>
> ## Outline
>
> 1. Setup
> 2. Original desert traveler — hand-rolled HP-style PCI
> 3. Weak-poison variant — hand-rolled PCI and closed-form Pearl PS
> 4. Cross-check with the framework's `ThinSearchSampler`

## 1. Setup

In [1]:
import itertools
import os
import warnings

import pyro
import pyro.distributions as dist
import torch

from pci.explanation.regime import condition_on_interventional_regime
from pci.explanation.scores import abs_diff_score
from pci.explanation.searchable import SearchableModel
from pci.explanation.thin_search import ThinSearchSampler

smoke_test = "CI" in os.environ
NUM_TS_SAMPLES = 100 if smoke_test else 4000

warnings.filterwarnings("ignore")
torch.manual_seed(0)

## 2. Original desert traveler — hand-rolled HP-style PCI

**SCM** (Pearl, *Causality* 2nd ed., p. 323):

$$c = P\,(u' \vee X'), \qquad d = X\,(u \vee P'), \qquad y = c \vee d.$$

**Factuals.** $X = 1$, $P = 1$, so $y^\star = 1$ in both noise states; the
factual mediator values are $(c, d) = (1, 0)$ at $u = 0$ and $(0, 1)$ at $u = 1$.

**PCI ingredients.** Suspects $\mathbf{S} = \{X, P\}$; candidate cause sets the
two singletons; witness pool $\mathbf{W} = \{c, d\}$ (four subsets); $\Delta$
deterministically flips $1 \to 0$. The AC-aligned kernel reduces, on this binary
deterministic model, to $\Phi > 0$ iff $Y$ flips when $\mathbf{C}$ is set to its
alternative and $\mathbf{T}$ is pinned at factual.

We enumerate the 4 (witness sets) × 2 (noise states) = 8 configurations per
cause and count flips.

In [2]:
# Structural equations for the original desert traveler.
# `c_pin` / `d_pin`: if provided, the mediator is held at the given value
# (witness pinning).
def desert_eqs(X, P, u, c_pin=None, d_pin=None):
    c = c_pin if c_pin is not None else P * max(1 - u, 1 - X)
    d = d_pin if d_pin is not None else X * max(u, 1 - P)
    y = max(c, d)
    return c, d, y


# PCI necessity expectation: fraction of (witness, noise) configurations
# in which the alternative intervention on `cause_var` flips y away from y*.
# Uniform Gamma over `witness_subsets`, uniform prior over `noise_states`.
def pci_necessity(cause_var, alt_value, factual, witness_subsets, noise_states):
    flips = 0
    total = 0
    for u in noise_states:
        f_c, f_d, f_y = desert_eqs(factual["X"], factual["P"], u)
        for w_set in witness_subsets:
            c_pin = f_c if "c" in w_set else None
            d_pin = f_d if "d" in w_set else None
            X_int = alt_value if cause_var == "X" else factual["X"]
            P_int = alt_value if cause_var == "P" else factual["P"]
            _, _, y_int = desert_eqs(X_int, P_int, u, c_pin=c_pin, d_pin=d_pin)
            flips += int(y_int != f_y)
            total += 1
    return flips / total, flips, total


witness_subsets = [(), ("c",), ("d",), ("c", "d")]

### Unconditional (uniform prior on $u$)

In [3]:
factual = {"X": 1, "P": 1}
noise_states = [0, 1]

for cause in ["X", "P"]:
    p, flips, total = pci_necessity(cause, 0, factual, witness_subsets, noise_states)
    target = 1/8
    status = "OK" if abs(p - target) < 1e-6 else "MISMATCH"
    print(f"PCI necessity, C = {{{cause}}}:  {p:.4f}  = {flips}/{total}     "
          f"(predicted {target:.4f} = 1/8)   {status}")

PCI necessity, C = {X}:  0.1250  = 1/8     (predicted 0.1250 = 1/8)   OK
PCI necessity, C = {P}:  0.1250  = 1/8     (predicted 0.1250 = 1/8)   OK


### Conditional on forensic evidence (no cyanide $\Rightarrow u = 1$)

In [4]:
for cause in ["X", "P"]:
    p, flips, total = pci_necessity(cause, 0, factual, witness_subsets, [1])
    target = {"X": 1/4, "P": 0.0}[cause]
    status = "OK" if abs(p - target) < 1e-6 else "MISMATCH"
    print(f"PCI necessity, C = {{{cause}}}:  {p:.4f}  = {flips}/{total}     "
          f"(predicted {target:.4f})   {status}")

PCI necessity, C = {X}:  0.2500  = 1/4     (predicted 0.2500)   OK
PCI necessity, C = {P}:  0.0000  = 0/4     (predicted 0.0000)   OK


**Result.** All four numbers match the proposal exactly: $\tfrac{1}{8}$ each
unconditionally, $\tfrac{1}{4}$ for $X$ and $0$ for $P$ after the forensic
report. Pearl's Def. 10.3.5 gives the same ranking with different units (1/2
each unconditionally, 1 and 0 after forensics).

## 3. Weak-poison variant — PCI and Pearl's PS

**SCM**: same as Pearl's original except the cyanide path now has a fatality
noise $\xi \sim \mathrm{Bern}(\alpha)$ with $\alpha = 0.1$ (small dose, mostly
non-fatal):

$$c = P\,(u' \vee X'), \quad V_C = c \cdot \xi, \quad
  d = X\,(u \vee P'), \quad y = V_C \vee d.$$

Two factual scenarios under the same observation $X=1, P=1, Y=1$:

- *Scenario A* — cyanide killed: forensic positive $\Rightarrow$ $u=0, \xi=1$
- *Scenario B* — dehydration killed: forensic negative $\Rightarrow$ $u=1$

We compute (i) PCI's necessity factor in each scenario and (ii) Pearl's $\mathrm{PS}$
for each cause in closed form. Sufficiency is where the methods diverge, and that
is what we want to read off.

In [5]:
ALPHA = 0.1


def weak_poison_eqs(X, P, u, xi, c_pin=None, d_pin=None, V_C_pin=None):
    c = c_pin if c_pin is not None else P * max(1 - u, 1 - X)
    d = d_pin if d_pin is not None else X * max(u, 1 - P)
    V_C = V_C_pin if V_C_pin is not None else c * xi
    y = max(V_C, d)
    return c, d, V_C, y


# `noise_states` is a list of (u, xi, weight) tuples.
def weak_pci_necessity(cause_var, alt_value, factual, witness_subsets, noise_states):
    f_X, f_P = factual["X"], factual["P"]
    flips_w = 0.0
    total_w = 0.0
    for u, xi, prob in noise_states:
        f_c, f_d, f_V_C, f_y = weak_poison_eqs(f_X, f_P, u, xi)
        for w_set in witness_subsets:
            c_pin = f_c if "c" in w_set else None
            d_pin = f_d if "d" in w_set else None
            V_C_pin = f_V_C if "V_C" in w_set else None
            X_int = alt_value if cause_var == "X" else f_X
            P_int = alt_value if cause_var == "P" else f_P
            _, _, _, y_int = weak_poison_eqs(
                X_int, P_int, u, xi, c_pin=c_pin, d_pin=d_pin, V_C_pin=V_C_pin
            )
            w = prob / len(witness_subsets)
            flips_w += w * int(y_int != f_y)
            total_w += w
    return flips_w / total_w


# All subsets of {c, d, V_C}
weak_witnesses = []
for r in range(0, 4):
    for combo in itertools.combinations(["c", "d", "V_C"], r):
        weak_witnesses.append(combo)
print(f"Number of candidate witness subsets: {len(weak_witnesses)}")

Number of candidate witness subsets: 8


### PCI necessity per scenario

In [6]:
posterior_A = [(0, 1, 1.0)]                            # Scenario A: forensic positive
posterior_B = [(1, 0, 1 - ALPHA), (1, 1, ALPHA)]        # Scenario B: forensic negative
factual = {"X": 1, "P": 1}

for label, posterior in [("A (cyanide killed)", posterior_A),
                         ("B (dehydration killed)", posterior_B)]:
    print(f"\nScenario {label}:")
    for cause in ["X", "P"]:
        p = weak_pci_necessity(cause, 0, factual, weak_witnesses, posterior)
        print(f"  PCI necessity, C = {{{cause}}}:  {p:.4f}")


Scenario A (cyanide killed):
  PCI necessity, C = {X}:  0.0000
  PCI necessity, C = {P}:  0.1250

Scenario B (dehydration killed):
  PCI necessity, C = {X}:  0.4875
  PCI necessity, C = {P}:  0.0000


The necessity factor agrees with the diagnosis: $P$ is the necessary cause in
Scenario A and $X$ is in Scenario B. (The magnitudes differ from the original
case because the witness pool is larger — 8 vs 4 subsets — so the per-witness
Γ-fraction is smaller.) Pearl Def. 10.3.5 makes the same diagnosis with full
posterior certainty in each scenario, so on necessity-only readings the two
methods agree.

### Pearl's PS in closed form

$$\mathrm{PS}(\bullet) \;=\; P\!\left(Y_{\bullet=1} = 1
  \,\middle|\, \bullet=0,\, Y=0\right),$$

with roots $X, P \sim \mathrm{Bern}(0.5)$ and $u \sim \mathrm{Bern}(0.5)$,
$\xi \sim \mathrm{Bern}(\alpha)$, all independent.

In [7]:
# Pearl's PS at population level: marginalise over the noise prior,
# condition on the cause being absent and the outcome being absent, then
# intervene to set the cause to 1 and report Pr(Y=1).
def closed_form_PS(cause_var, alpha=ALPHA):
    cond_total = 0.0
    cond_outcome = 0.0
    for X_val, P_val, u, xi in itertools.product([0, 1], repeat=4):
        if cause_var == "X" and X_val != 0: continue
        if cause_var == "P" and P_val != 0: continue
        prob = 0.5 * 0.5 * 0.5 * (alpha if xi == 1 else 1 - alpha)
        _, _, _, y_obs = weak_poison_eqs(X_val, P_val, u, xi)
        if y_obs != 0: continue
        cond_total += prob
        X_int = 1 if cause_var == "X" else X_val
        P_int = 1 if cause_var == "P" else P_val
        _, _, _, y_int = weak_poison_eqs(X_int, P_int, u, xi)
        if y_int == 1:
            cond_outcome += prob
    return cond_outcome / cond_total


for cause in ["X", "P"]:
    ps = closed_form_PS(cause)
    target = {"X": 0.76, "P": 0.1}[cause]
    print(f"PS({cause}) = {ps:.4f}  (proposal target: {target})")

PS(X) = 0.7632  (proposal target: 0.76)
PS(P) = 0.1000  (proposal target: 0.1)


**Result.** $\mathrm{PS}(P) = 0.1$ exactly (the cyanide-fatality rate $\alpha$);
$\mathrm{PS}(X) = 0.7632$ (matches the $\approx 0.76$ in the proposal).
A composite $ci$ that combines necessity and sufficiency therefore ranks the
shooter in Scenario B substantially above the poisoner in Scenario A, even though
Def. 10.3.5 calls both "$P(\text{caused}) = 1$." This is the sufficiency
discrimination the proposal claims.

## 4. Cross-check with the framework's `ThinSearchSampler`

For completeness we also run the framework's `ThinSearchSampler` on the original
desert traveler.

**Caveat.** The framework's witness mechanism pins *non-deterministic*
sites — i.e., the model's root variables — at their factual values, not
*mediator* variables. The paper's HP-style witness mechanism (used in
Sections 2 and 3 above) treats mediators $c, d$ as valid witnesses, which is
necessary to make $X$ flip $Y$ in the cyanide-vs-dehydration setup. So the
framework's per-suspect numbers will differ in magnitude from the hand-rolled
$\tfrac{1}{8}$ — they're measuring a related but distinct quantity. This is
useful to note when designing a future PCI implementation that supports
mediator pinning.

In [8]:
BATCH_SIZE = 2  # idx 0: u=0 (cyanide); idx 1: u=1 (dehydration)


def desert_model(kwargs_iterable=None):
    if kwargs_iterable is None:
        kwargs_iterable = [{"observations_dict": None, "n_size": BATCH_SIZE}, {}, {}]
    batch_size = kwargs_iterable[0]["n_size"]
    logits = torch.ones(batch_size, 1, 1, 2)
    u = pyro.sample("u", dist.Categorical(logits=logits))
    X = pyro.sample("X", dist.Categorical(logits=logits))
    P = pyro.sample("P", dist.Categorical(logits=logits))
    uf, Xf, Pf = u.float(), X.float(), P.float()
    c = pyro.deterministic("c", Pf * torch.maximum(1.0 - uf, 1.0 - Xf), event_dim=0)
    d = pyro.deterministic("d", Xf * torch.maximum(uf, 1.0 - Pf), event_dim=0)
    y = pyro.deterministic("y", torch.maximum(c, d), event_dim=0)
    return {
        "categorical": {"u": u, "X": X, "P": P},
        "continuous": {"c": c, "d": d, "y": y},
    }


def _make_int(values):  return torch.tensor(values, dtype=torch.long).view(BATCH_SIZE, 1, 1)
def _make_float(values): return torch.tensor(values, dtype=torch.float).view(BATCH_SIZE, 1, 1)


factual_struct = {
    "categorical": {
        "u": _make_int([0, 1]),
        "X": _make_int([1, 1]),
        "P": _make_int([1, 1]),
    },
    "continuous": {
        "c": _make_float([1.0, 0.0]),
        "d": _make_float([0.0, 1.0]),
        "y": _make_float([1.0, 1.0]),
    },
}

searchable = SearchableModel(
    structured_model=desert_model,
    sites_of_interest=["u", "X", "P", "c", "d", "y"],
    suspects=["X", "P"],
    deterministic_sites=["c", "d", "y"],
    shared_noise_sites=["u"],
    outcome_variable="y",
)

sampler = ThinSearchSampler(
    structured_model=searchable,
    conditioned_alternatives=False,
    factual_exclusion=True,
    max_antecedents=2,
    max_witnesses_dropped=1,
)
print(f"Suspects:  {sampler.suspects}")
print(f"Witnesses: {sampler.witnesses}  (mediators c, d not included)")

Suspects:  ['X', 'P']
Witnesses: ['u', 'X', 'P']  (mediators c, d not included)


In [9]:
results = sampler.sample(factual_struct, num_samples=NUM_TS_SAMPLES)
print(f"\nGot {NUM_TS_SAMPLES} samples; necessity['y'] shape: "
      f"{tuple(results.necessity['y'].shape)}")

for v in ["X", "P"]:
    conditioned = condition_on_interventional_regime(
        results_dictionary=results,
        reference_variable_names=[v],
        antecedent_regimes={v: True},
    )
    y_nec = conditioned["regime_necessity"]["y"].detach()
    y_suff = conditioned["regime_sufficiency"]["y"].detach()
    y_fact = factual_struct["continuous"]["y"]
    sc = abs_diff_score(factual_outcomes=y_fact,
                         suff_outcomes=y_suff,
                         nec_outcomes=y_nec)
    print(f"\nFramework score for {v}:")
    print(f"  necessity   per batch: {sc['nec'][:, :, 0, 0].nanmean(dim=0).tolist()}")
    print(f"  sufficiency per batch: {sc['suff'][:, :, 0, 0].nanmean(dim=0).tolist()}")
    print(f"  necessity overall: {sc['nec'].nanmean().item():.4f}")

  0%|          | 0/4000 [00:00<?, ?it/s]

  2%|▏         | 66/4000 [00:00<00:06, 651.81it/s]

  4%|▎         | 140/4000 [00:00<00:05, 700.36it/s]

  5%|▌         | 213/4000 [00:00<00:05, 710.56it/s]

  7%|▋         | 287/4000 [00:00<00:05, 720.89it/s]

...

 96%|█████████▌| 3846/4000 [00:07<00:00, 740.84it/s]

 98%|█████████▊| 3922/4000 [00:07<00:00, 746.06it/s]

100%|█████████▉| 3999/4000 [00:07<00:00, 750.38it/s]

100%|██████████| 4000/4000 [00:07<00:00, 541.50it/s]


Got 4000 samples; necessity['y'] shape: (4000, 2, 1, 1)

Framework score for X:
  necessity   per batch: [0.698113203048706, 0.6964285969734192]
  sufficiency per batch: [0.0, 0.0]
  necessity overall: 0.6973

Framework score for P:
  necessity   per batch: [0.6879761219024658, 0.6886386275291443]
  sufficiency per batch: [0.0, 0.0]
  necessity overall: 0.6883


**Reading the framework output.** The framework reports much higher necessity
($\approx 0.7$ for each cause) than the HP-style hand-rolled $\tfrac{1}{8}$.
The reason is in the caveat above: without mediator pinning, intervening
$X \to 0$ allows the cyanide path to fire (because $P$ is sampled fresh in the
necessity world and frequently becomes $0$, or because the partially-pinned
witnesses don't block the relevant path). What the framework computes is
correct *for the witness mechanism it implements*; the implementation just
does not yet match the broader $\mathbf{W} \subseteq \mathbf{V}$ specification
the paper assumes. The hand-rolled enumeration in §§2–3 reproduces the proposal's
numbers because it pins mediators as witnesses, matching the paper's spec.

## Summary

| Claim in the proposal | Verified value |
|---|---|
| PCI necessity for $\{X\}$ on original DT, uniform prior | $1/8 = 0.1250$ |
| PCI necessity for $\{P\}$ on original DT, uniform prior | $1/8 = 0.1250$ |
| PCI necessity for $\{X\}$ on original DT, given $\neg$cyanide | $1/4 = 0.2500$ |
| PCI necessity for $\{P\}$ on original DT, given $\neg$cyanide | $0$ |
| Pearl's $\mathrm{PS}(P)$ in weak-poison variant | $0.1$ exactly |
| Pearl's $\mathrm{PS}(X)$ in weak-poison variant | $0.7632 \approx 0.76$ |

All match. The proposal's quantitative claims are sound.

**Implementation note.** The current `ThinSearchSampler` does not pin mediator
variables as witnesses, only co-root variables. The hand-rolled enumeration in
this notebook matches what the paper's PCI specification calls for; extending
the framework to support mediator witnesses would make its output align with
the hand-rolled numbers on this canonical example.